# 🛒 Supermarket Sales & Stock Insight Tool
### Academic Mini Project — Python Data Analytics Engine

---

## 📌 Overview & PRD Compliance
- **Purpose**: Fast, syllabus-compliant retail decision analytics for supermarket managers.
- **Core Constraint**: No external web frameworks (no Flask/Django/HTML/JS), no OOP classes, no external plotting libraries (matplotlib/seaborn).
- **Syllabus Topics Covered**: Variables, Operators, Strings, Lists, Tuples, Sets, Dictionaries, Functions, Files, NumPy, Pandas.
- **Visualization**: Custom ASCII text bar charts constructed purely with string manipulation.
- **Environment**: Runs seamlessly in Google Colab, Linux, macOS, and Windows terminals.


## ⚙️ Step 1: Environment Setup & Module Imports
Ensure the repository directory and `supermarket_insight` package are in Python's module path.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add current directory and supermarket_insight to sys.path
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

pkg_dir = os.path.join(project_root, 'supermarket_insight')
if os.path.exists(pkg_dir) and pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)

print('Python Version :', sys.version.split()[0])
print('Working Dir    :', project_root)
print('Environment initialized successfully.')


## 📂 Step 2: Dataset Detection / Loading
Locates `supermarket_sales.csv` in local paths or prompts for upload if running in a fresh Colab instance.

In [ ]:
candidate_paths = [
    'supermarket_insight/data/supermarket_sales.csv',
    'data/supermarket_sales.csv',
    'supermarket_sales.csv',
    '/content/supermarket_sales.csv',
    '/content/supermarket_insight/data/supermarket_sales.csv'
]

data_file = None
for p in candidate_paths:
    if os.path.exists(p):
        data_file = p
        break

if data_file:
    print(f'Dataset located at: {data_file}')
else:
    print('Dataset not found locally. If on Colab, please upload supermarket_sales.csv:')
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.csv'):
                data_file = fname
                break
    except Exception as e:
        print(f'Upload prompt skipped: {e}')


## 🧹 Step 3: Data Ingestion, Cleaning & Health Check (PRD F1 & F2)
- Strips leading/trailing whitespace and standardizes title casing.
- Drops duplicate invoice IDs.
- Derives temporal columns (`Month`, `Hour`).
- Reports missing values and health metrics safely without crashing.

In [ ]:
from supermarket_insight import store, report

raw_df = store.load_data(data_file)
clean_df, health_stats = store.clean_data(raw_df)

# Display formatted health check
report.display_health_check(health_stats)

print(f'\nCleaned DataFrame preview ({len(clean_df):,} records, {clean_df.shape[1]} columns):')
clean_df[['Invoice ID', 'Branch', 'City', 'Product line', 'Quantity', 'Total', 'Month', 'Hour']].head(5)


## 📊 Step 4: Multi-Dimensional Revenue Views (PRD F3)
Aggregates financial performance across:
1. **Branch** (Branches A, B, C)
2. **Product Line**
3. **Month** (January, February, March)
4. **Peak Hour of Day** (Busiest shopping hours)

In [ ]:
report.display_revenue_views(clean_df)


## 🏆 Step 5: Best & Slowest Performing Products (PRD F4)
Ranked by both total monetary revenue and physical units sold.

In [ ]:
report.display_product_rankings(clean_df)


## 💳 Step 6: Payment Method Share (PRD F5)
Calculates transaction percentage and monetary share across Cash, Ewallet, and Credit Card.

In [ ]:
report.display_payment_share(clean_df)


## ⚠️ Step 7: Low-Sales Velocity Alerts (PRD F6)
- Detects products falling short of a customizable target threshold.
- You can adjust `ALERT_TARGET` below to any dollar amount.

In [ ]:
# Configurable runtime threshold (Default $5,000. Try setting to $52,000 to trigger alerts)
ALERT_TARGET = 52000.0

report.display_low_sales_alerts(clean_df, limit=ALERT_TARGET)


## 🔄 Step 8: Branch Product Gap Analysis — Set Theory (PRD F7)
Applies mathematical set operations:
- **Set Intersection**: Products present in every branch catalog.
- **Set Difference**: Exclusive products stocked in one branch but missing in another.

In [ ]:
report.display_branch_gap(clean_df)


## 📈 Step 9: Bill Statistics — NumPy Integration (PRD F8)
Uses NumPy vectorized operations on transaction amounts:
- Mean, Median, Standard Deviation, 90th Percentile, Minimum, and Maximum.

In [ ]:
report.display_bill_statistics(clean_df)


## 🔍 Step 10: Multi-Column Tolerant Search Engine (PRD F9)
Performs case-insensitive, whitespace-tolerant substring matching across product lines, branches, cities, dates, and invoice IDs.

In [ ]:
from supermarket_insight import search

# Test queries: 'health and beauty', 'credit card', 'branch a', 'electronic'
query_string = '  health   and   BEAUTY  '
results = search.search_records(clean_df, query_string)

report.display_search_results(results, query_string)


## 📊 Step 11: Visual Text Bar Charts (PRD F10)
ASCII terminal bar charts generated purely with string multiplication—zero external plotting dependencies.

In [ ]:
report.display_bar_charts(clean_df)


## 💾 Step 12: Export Deliverables (PRD F11)
Generates and saves `monthly_report.csv` and formatted `report.txt`.

In [ ]:
from supermarket_insight import config

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
csv_out = os.path.join(config.OUTPUT_DIR, 'monthly_report.csv')
txt_out = os.path.join(config.OUTPUT_DIR, 'report.txt')

csv_ok, txt_ok = report.export_all_reports(
    clean_df,
    low_sales_limit=ALERT_TARGET,
    csv_path=csv_out,
    txt_path=txt_out
)

print(f'✅ Monthly aggregated CSV exported: {csv_out} (Success: {csv_ok})')
print(f'✅ Executive text summary exported : {txt_out} (Success: {txt_ok})')

# If in Google Colab, offer download triggers
try:
    from google.colab import files
    print('Downloading files to your local machine...')
    files.download(csv_out)
    files.download(txt_out)
except Exception:
    print('Files are saved in your workspace at supermarket_insight/data/output/')


## 🧪 Step 13: Run Automated Test Suite (PRD Section 13)
Executes all 10 verification tests using plain `assert` statements.

In [ ]:
from supermarket_insight.tests import run_tests

test_success = run_tests.run_all_tests()
assert test_success, 'One or more automated tests failed!'


## 🎮 Step 14: Interactive Menu Loop (PRD F12)
Run the interactive numbered console interface with the dictionary dispatcher.
*Tip: Type 0 to exit back to the notebook.*

In [ ]:
# Uncomment and run the following line to launch the full interactive CLI menu in the notebook
# from supermarket_insight import main
# main.main()
